This notebook demonstrates how to use the BiomechZoo library to analyze movement data collected from IMU sensors.
The analysis includes data preprocessing, event detection, gait cycle partitioning, normalization, and visualization of the results.

Data source: The data used in this notebook is from the "Losing" dataset, which contains IMU data collected from participants performing
various walking tasks. The data were modified for the purposes of this tutorial.

The full dataset is available at: https://doi.org/10.6084/m9.figshare.c.5758997

The dataset used here is avaialbe in https://github.com/mcgillmotionlab/biomechzoo/tree/main/data/losing/raw

The dataset is described in the following paper: Losing, V., Hasenjäger, M. A Multi-Modal Gait Database of Natural Everyday-Walk in an Urban Environment. Sci Data 9, 473 (2022). https://doi.org/10.1038/s41597-022-01580-3

In [55]:
# import relevant biomech modules
%load_ext autoreload
%autoreload 2
import os
from biomechzoo.biomechzoo import BiomechZoo
from biomechzoo.utils.find_repo_root import find_repo_root

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
# Step 0: get raw data folder and initialize biomechzoo object
project_root = find_repo_root(os.getcwd())
fld_raw_data = os.path.join(project_root, 'data', 'losing', 'raw')
inplace = False  # tells biomechzoo to create new folders instead of writing to the original folder
verbose = 0 # set verbisity level
name_excludes = ['section_1', 'section_3'] # these trials won't be processed
bmech = BiomechZoo(fld_raw_data, verbose=verbose, inplace=inplace)

In [57]:
# step 1: convert to zoo
# frequency is 60 Hz as per the database documentation
bmech.table2zoo(extension='csv', freq=60, out_folder='1-csv2zoo')

In [58]:
# step 2 clean
ch_remove = ['walk_mode', 'walk_orientation','time_orig']
ch_current = ['jointAngle_jRightKnee_x', 'jointAngle_jRightKnee_y', 'jointAngle_jRightKnee_z']
ch_new = ['RightKneeAngle_x', 'RightKneeAngle_y', 'RightKneeAngle_z']
bmech.removechannel(ch=ch_remove, out_folder='2-clean')
bmech.renamechannel(ch_current=ch_current, ch_new=ch_new, inplace=True)

In [59]:
# inspect the file
bmech.file_info()

File name: id16_courseA_50cycles.zoo
File path: /Users/phild/Code/biomechzoo/data/losing/2-clean
Size: 1065552 bytes
Video frequency: 60 Hz
Analog frequency: {} Hz
Number of channels: 40
Events: 0 across 0 of 40 channels

Channel preview (first 5 rows):

   time  acceleration_Pelvis_x  acceleration_Pelvis_y  acceleration_Pelvis_z  \
0     0              -2.336372               6.582549               5.247364   
1    16              -0.424324               0.079229               3.618380   
2    33              -1.300589              -1.217998               1.578247   
3    50              -4.053475              -4.308287              -0.973026   
4    66              -2.177729              -1.211102               0.927906   

   acceleration_RightLowerLeg_x  acceleration_RightLowerLeg_y  \
0                     -0.132983                     -4.775965   
1                     -3.594236                    -12.496243   
2                    -16.211113                     -4.161099   
3   

,time,acceleration_Pelvis_x,acceleration_Pelvis_y,acceleration_Pelvis_z,acceleration_RightLowerLeg_x,acceleration_RightLowerLeg_y,acceleration_RightLowerLeg_z,acceleration_RightFoot_x,acceleration_RightFoot_y,acceleration_RightFoot_z,...,angularVelocity_LeftFoot_z,insoles_RightFoot_is_step,insoles_LeftFoot_is_step,insoles_RightFoot_is_lifted,insoles_LeftFoot_is_lifted,insoles_LeftFoot_on_ground,insoles_RightFoot_on_ground,RightKneeAngle_x,RightKneeAngle_y,RightKneeAngle_z
0,0,-2.336372,6.582549,5.247364,-0.132983,-4.775965,1.328061,-4.594653,0.063161,1.812828,...,-0.521989,1,0,0,0,1,1,4.175873,8.551077,19.610740
1,16,-0.424324,0.079229,3.618380,-3.594236,-12.496243,-4.557032,0.753056,-6.201272,-2.844540,...,-0.713763,0,0,0,0,1,1,3.980738,5.287679,23.393889
2,33,-1.300589,-1.217998,1.578247,-16.211113,-4.161099,-3.474461,-30.055765,0.673723,4.085116,...,-0.380541,0,0,0,0,1,1,2.633581,2.408448,25.970458
3,50,-4.053475,-4.308287,-0.973026,-11.202896,-9.874950,1.348414,2.334383,-4.637576,-0.186800,...,-0.059054,0,0,0,0,1,1,1.695268,0.785412,27.642093
4,66,-2.177729,-1.211102,0.927906,-6.929343,-5.389872,-0.816749,1.334546,-5.640506,0.183604,...,0.194073,0,0,0,0,1,1,0.268107,-0.149329,28.677854


In [60]:
# step 3: get foot strikes from mcgrath algorithm
#bmech.compute_magnitude(ch_1='acceleration_RightLowerLeg_x', ch_2='acceleration_RightLowerLeg_y', ch_3='acceleration_RightLowerLeg_z', ch_new='acceleration_RightLowerLeg_magnitude', out_folder='3-processing', inplace=False)

bmech.addevent(event_type='mcgrath_fs', event_name='foot_strike', ch='angularVelocity_RightLowerLeg_x', out_folder='3-processing', inplace=True)

In [61]:
# step 4: partition by gait cycle and normalize
bmech.split_trial_by_gait_cycle(first_event_name='foot_strike_1', out_folder='4-split', inplace=False)
bmech.normalize(inplace=True)

In [62]:
# step 7: plot mean and standard deviation across all files
from biomechzoo.ensembler.ensembler import Ensembler
from biomechzoo.ensembler.plot_spec import PlotSpec
from biomechzoo.ensembler.renderers import MeanSDRenderer
from biomechzoo.ensembler.helpers import ConditionSpec, ConditionSource
from biomechzoo.ensembler.renderers import IndividualLinesRenderer, EventOverlayRenderer, CompositeRenderer
subj_list = ['id14', 'id16', 'id19']

channel = ['RightKneeAngle_z']

spec = ConditionSpec(
    source=ConditionSource.BETWEEN,
    conditions=['courseA', 'courseB', 'courseC'],
)

lines_and_events = CompositeRenderer(IndividualLinesRenderer(), EventOverlayRenderer())     # within stuff

fig = (
    Ensembler(in_folder=bmech.in_folder,  channels=channel, n_rows=1,  n_cols=2, subj_list=subj_list, condition_spec=spec)
    .add_subplot(PlotSpec(channel=channel[0], condition = 'courseA', companions = ['courseB', 'courseC'], row=1, col=1, renderer=IndividualLinesRenderer()))
    .add_subplot(PlotSpec(channel=channel[0], condition = 'courseA', companions = ['courseB', 'courseC'], row=1, col=2, renderer=MeanSDRenderer()))
    .build(title='Joint angles across courses')
)
fig.show()
fig.write_html('walking_analysis.html')